In [1]:
start_date = "2023-09-01"
end_date   = "2023-09-30"

In [2]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

params = {
    "latitude": 6.9355,
    "longitude": 79.8487,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ["temperature_2m", "precipitation"],
    "timezone": "auto"
}

responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
response = responses[0]

hourly = response.Hourly()
hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    ),
    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "precipitation": hourly.Variables(1).ValuesAsNumpy()
}

hourly_df = pd.DataFrame(hourly_data)
hourly_df["date"] = hourly_df["date"].dt.tz_convert("Asia/Colombo")


In [4]:
daily_df = (
    hourly_df
    .groupby(hourly_df["date"].dt.date)
    .agg(
        T_max=("temperature_2m", "max"),
        T_min=("temperature_2m", "min"),
        Avg_temp=("temperature_2m", "mean"),
        RF=("precipitation", "sum")
    )
    .reset_index()
    .rename(columns={"date": "Date"})
)
daily_df

,Date,T_max,T_min,Avg_temp,RF
0,2023-09-01,28.006500,24.256500,25.452333,23.900002
1,2023-09-02,27.806499,24.356501,25.662750,24.000000
2,2023-09-03,27.606501,24.506500,25.635666,32.400002
3,2023-09-04,28.906500,24.506500,26.423166,23.700001
4,2023-09-05,28.306499,24.356501,26.187750,20.400000
5,2023-09-06,27.156500,24.106501,25.360666,18.200001
6,2023-09-07,27.206501,24.206501,25.375250,23.900000
7,2023-09-08,28.506500,24.206501,26.014833,18.700001
8,2023-09-09,28.356501,24.256500,26.143999,14.700000
9,2023-09-10,27.806499,24.406500,25.929419,36.400002


In [6]:
existing_df = pd.read_csv("../Data/weather_daily_2013_2023.csv")
existing_df["Date"] = pd.to_datetime(existing_df["Date"])
mask = (existing_df["Date"] >= start_date) & (existing_df["Date"] <= end_date)
existing_range_df = existing_df.loc[mask, ["Date", "RF", "T max ", "T min", "Avg_tem"]]


In [8]:
existing_range_df = existing_range_df.rename(columns={
    
    'T max ': 'T max'
})

In [9]:
daily_df["Date"] = pd.to_datetime(daily_df["Date"]).dt.date
existing_range_df["Date"] = pd.to_datetime(existing_range_df["Date"]).dt.date

In [14]:
existing_range_df = existing_range_df.rename(columns={
    "T max": "T_max_orig",
    "T min": "T_min_orig",
    "Avg_tem": "Avg_temp_orig",
    "RF": "RF_orig"
})

daily_df = daily_df.rename(columns={
    "T_max": "T_max_api",
    "T_min": "T_min_api",
    "Avg_temp": "Avg_temp_api",
    "RF": "RF_api"
})


In [15]:
compare_df = daily_df.merge(existing_range_df, on="Date", suffixes=("_api", "_orig"))

print(compare_df.head())


         Date  T_max_api  T_min_api  Avg_temp_api     RF_api  RF_orig  \
0  2023-09-01  28.006500  24.256500     25.452333  23.900002     33.1   
1  2023-09-02  27.806499  24.356501     25.662750  24.000000     74.3   
2  2023-09-03  27.606501  24.506500     25.635666  32.400002     48.7   
3  2023-09-04  28.906500  24.506500     26.423166  23.700001     13.4   
4  2023-09-05  28.306499  24.356501     26.187750  20.400000     12.6   

   T_max_orig  T_min_orig  Avg_temp_orig  
0        28.6        24.5          26.55  
1        29.3        24.8          27.05  
2        28.5        24.9          26.70  
3        31.2        24.1          27.65  
4        30.1        24.8          27.45  


In [19]:
from sklearn.metrics import mean_absolute_error

results = []
for col_pair in [
    ("T_max_api", "T_max_orig"),
    ("T_min_api", "T_min_orig"),
    ("Avg_temp_api", "Avg_temp_orig"),
    ("RF_api", "RF_orig")
]:
    api_col, orig_col = col_pair
    mae = mean_absolute_error(compare_df[orig_col], compare_df[api_col])
    corr = compare_df[[orig_col, api_col]].corr().iloc[0, 1]
    results.append({"Metric": orig_col, "MAE": mae, "Correlation": corr})

results_df = pd.DataFrame(results)
print(results_df)


          Metric        MAE  Correlation
0     T_max_orig   1.943933     0.503408
1     T_min_orig   0.689433     0.380598
2  Avg_temp_orig   1.513292     0.531104
3        RF_orig  18.073333     0.270876
